# Predict Wine Quality with Regularization – Solution

**Short name (GitHub):** `WineReg`  
**Lab source:** Codecademy *Predict Wine Quality with Regularization* (UCI red-wine table).  
**Data:** `data/wine_quality.csv` (1,599 bottles × 11 chemistry tests + binary `quality`).  
**Companion files:** `WineReg_Solution.ipynb`, `WineReg_Reusable_Template.ipynb`, `WineReg.py`, `WineReg_Cheatsheet.docx`, `WineReg_Project_Memo.docx`, `WineReg_Strategy_Guide.docx`, `WineReg_1Page_Summary_Report.docx`, `wine_reg_flowchart.png`.

Worked answers for the practice skeleton. Numbers are from sklearn 1.x with `random_state=99`.

**Target.** Original ratings were 1–10. Here `quality = 1` means a *good* bottle (rating > 5) and `0` means *bad* (≤ 5). You will:

1. Fit an unregularized logistic classifier and read the coefficient bar.
2. Confirm that default L2 (`C=1`) barely shrinks anything.
3. Tune ridge `C` with a coarse grid, then `GridSearchCV`.
4. Use L1 (`LogisticRegressionCV`) as a feature-selection method — `density` should go to zero.


## Inline cheat-sheet (keep this cell visible)

See also **`WineReg_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Scale | `StandardScaler().fit(features)` then `.transform(features)` |
| Split | `train_test_split(X, y, test_size=0.2, random_state=99)` |
| No penalty | `LogisticRegression(penalty=None, max_iter=2000)` (lesson: `'none'`; future: `C=np.inf`) |
| Default ridge | `LogisticRegression()` historically L2 with `C=1` |
| F1 | `f1_score(y_true, y_pred)` — harmonic mean of precision & recall |
| Coarse C | `[0.0001, 0.001, 0.01, 0.1, 1]` — smaller C = stronger L2 |
| Fine C | `np.logspace(-4, -2, 100)` |
| Grid search | `GridSearchCV(clf, {'C': C_array}, scoring='f1', cv=5)` |
| L1 CV | `LogisticRegressionCV(Cs=..., penalty='l1', solver='liblinear', scoring='f1')` |
| Coefs | `pd.Series(clf.coef_.ravel(), predictors).sort_values()` |
| C meaning | `C = 1 / λ`. Tiny C → shrink toward 0. Huge C → unregularized. |

**sklearn ≥1.8 note.** `penalty` is deprecated. Lesson code with `penalty='none'` still runs; prefer `penalty=None` or `C=np.inf` for no regularization, and set `penalty='l2'` / `'l1'` explicitly in this notebook so the intent stays readable.


## Flowchart of the desired outcome

![WineReg flow](wine_reg_flowchart.png)

Scale first (alcohol lives on a different scale than density). Split once and freeze `random_state=99`. Use F1, not raw accuracy — a 53.5% “good” prior makes accuracy a weak headline. Validate the “best” ridge model on the original test fold, then switch to L1 for sparsity.


## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report


## 1. Load the table and separate X / y

In [ ]:
df = pd.read_csv("data/wine_quality.csv")
print("shape", df.shape)
print(df.columns.tolist())
print(df["quality"].value_counts())
print("good rate", float(df["quality"].mean()))
display(df.head())
y = df["quality"]
features = df.drop(columns=["quality"])
print("features", list(features.columns))


## 2. Task 1 — scale

In [ ]:
standard_scaler_fit = StandardScaler().fit(features)
X = standard_scaler_fit.transform(features)
print("X shape", X.shape, "column means", np.round(X.mean(axis=0)[:4], 4), "…")
print("column sds", np.round(X.std(axis=0)[:4], 4), "…")


## 3. Task 2 — split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=99
)
print("train", X_train.shape, "test", X_test.shape)
print("train good rate", float(y_train.mean()), "test good rate", float(y_test.mean()))


## 4. Task 3 — unregularized LR

In [ ]:
clf_no_reg = LogisticRegression(penalty=None, max_iter=2000)
clf_no_reg.fit(X_train, y_train)
print("intercept", float(clf_no_reg.intercept_[0]))
print(pd.Series(clf_no_reg.coef_.ravel(), index=features.columns).sort_values())


## 5. Task 4 — coefficient bar

In [ ]:
predictors = features.columns
coefficients = clf_no_reg.coef_.ravel()
coef = pd.Series(coefficients, predictors).sort_values()
ax = coef.plot(kind="barh", title="Coefficients (no regularization)")
plt.tight_layout()
plt.show()
plt.clf()
print(coef.round(3))
# Expected: alcohol (+) largest; sulphates (+); total SO2 and volatile acidity (−).


## 6. Task 5 — F1

In [ ]:
y_pred_test = clf_no_reg.predict(X_test)
y_pred_train = clf_no_reg.predict(X_train)
print("Training Score", f1_score(y_train, y_pred_train))
print("Testing Score", f1_score(y_test, y_pred_test))
print("Training acc", accuracy_score(y_train, y_pred_train))
print("Testing acc", accuracy_score(y_test, y_pred_test))
# Expected F1 ≈ 0.773 train / 0.727 test


## 7. Task 6 — default LR

In [ ]:
clf_default = LogisticRegression(max_iter=2000)  # lesson: LogisticRegression()
clf_default.fit(X_train, y_train)
print("params C, penalty", clf_default.get_params()["C"], clf_default.get_params()["penalty"])


## 8. Task 7 — ridge F1

In [ ]:
y_pred_train_ridge = clf_default.predict(X_train)
y_pred_test_ridge = clf_default.predict(X_test)
print("Ridge-regularized Training Score", f1_score(y_train, y_pred_train_ridge))
print("Ridge-regularized Testing Score", f1_score(y_test, y_pred_test_ridge))
# Expected: essentially identical to the unregularized scores.
# C=1 is a loose constraint — the unpenalized minimum still sits inside the ball.


## 9. Task 8 — coarse C

In [ ]:
training_array = []
test_array = []
C_array = [0.0001, 0.001, 0.01, 0.1, 1]
for x in C_array:
    clf = LogisticRegression(C=x, max_iter=2000)
    clf.fit(X_train, y_train)
    training_array.append(f1_score(y_train, clf.predict(X_train)))
    test_array.append(f1_score(y_test, clf.predict(X_test)))
print("C     ", C_array)
print("train ", [round(v, 4) for v in training_array])
print("test  ", [round(v, 4) for v in test_array])
# Expected test F1 peaks near C=0.001 (≈0.741) then drifts down toward C=1 (≈0.727).


## 10. Task 9 — plot C

In [ ]:
plt.plot(C_array, training_array, marker="o", label="Training Score")
plt.plot(C_array, test_array, marker="s", label="Test Score")
plt.xscale("log")
plt.xlabel("C")
plt.ylabel("F1")
plt.title("Coarse L2 grid")
plt.legend()
plt.show()
plt.clf()


## 11. Task 10 — grid

In [ ]:
C_array = np.logspace(-4, -2, 100)
tuning_C = {"C": C_array}
print("C window", C_array.min(), "→", C_array.max(), "n=", len(C_array))


## 12. Task 11 — GridSearchCV

In [ ]:
clf_gs = LogisticRegression(max_iter=2000)
gs = GridSearchCV(clf_gs, param_grid=tuning_C, scoring="f1", cv=5)
gs.fit(X_train, y_train)
print("fitted", len(C_array), "candidates × 5 folds")


## 13. Task 12 — best C

In [ ]:
print(gs.best_params_, gs.best_score_)
# Expected C ≈ 0.00206, mean CV F1 ≈ 0.773


## 14. Task 13 — hold-out validation

In [ ]:
clf_best = LogisticRegression(C=gs.best_params_["C"], max_iter=2000)
clf_best.fit(X_train, y_train)
y_pred_best = clf_best.predict(X_test)
print("hold-out F1", f1_score(y_test, y_pred_best))
print("hold-out acc", accuracy_score(y_test, y_pred_best))
print("hold-out AUC", roc_auc_score(y_test, clf_best.predict_proba(X_test)[:, 1]))
print(classification_report(y_test, y_pred_best, digits=3))
# Expected hold-out F1 ≈ 0.735, AUC ≈ 0.811


## 15. Task 14 — L1 CV

In [ ]:
C_array = np.logspace(-2, 2, 100)
clf_l1 = LogisticRegressionCV(
    Cs=C_array, cv=5, penalty="l1", scoring="f1", solver="liblinear", max_iter=2000
)
clf_l1.fit(X, y)
print("L1 search done")


## 16. Task 15 — L1 C and coefs

In [ ]:
print("Best C value", clf_l1.C_)
print("Best fit coefficients")
print(pd.Series(clf_l1.coef_.ravel(), index=predictors).round(5))
# Expected C ≈ 0.26; density coefficient ≈ 0.


## 17. Task 16 — L1 bar

In [ ]:
coefficients = clf_l1.coef_.ravel()
coef = pd.Series(coefficients, predictors).sort_values()
plt.figure(figsize=(10, 5))
coef.plot(kind="barh", title="Coefficients for tuned L1")
plt.tight_layout()
plt.show()
plt.clf()


## 18. Task 17 — dropped feature

In [ ]:
zeroed = [name for name, v in zip(predictors, clf_l1.coef_.ravel()) if abs(v) < 1e-10]
print("zeroed by L1:", zeroed)
# density. It is almost a linear combination of alcohol, residual sugar and fixed acidity,
# so the L1 path can drop it without giving up much F1.


## 19. Alternate code (same scientific result)

Worked alternatives you can swap into the lesson cells.


In [ ]:
# A. Unregularized via C = inf (sklearn ≥1.8 preferred form)
clf_inf = LogisticRegression(C=np.inf, max_iter=2000)
clf_inf.fit(X_train, y_train)
print("C=inf test F1", f1_score(y_test, clf_inf.predict(X_test)))

# B. Pipeline so the scaler cannot leak if you ever change the split
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(C=0.002, penalty="l2", max_iter=2000)),
])
# pipeline wants the *raw* features, not X already scaled
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    features, y, test_size=0.2, random_state=99
)
pipe.fit(Xr_train, yr_train)
print("pipeline test F1", f1_score(yr_test, pipe.predict(Xr_test)))

# C. L2 path with LogisticRegressionCV instead of GridSearchCV
clf_l2cv = LogisticRegressionCV(
    Cs=np.logspace(-4, -2, 40), cv=5, penalty="l2", scoring="f1", max_iter=2000
)
clf_l2cv.fit(X_train, y_train)
print("L2-CV best C", clf_l2cv.C_, "test F1",
      f1_score(y_test, clf_l2cv.predict(X_test)))

# D. Coefficient table instead of a bar
pd.DataFrame({
    "feature": predictors,
    "unreg": clf_no_reg.coef_.ravel(),
    "ridge_C1": clf_default.coef_.ravel(),
    "l1": clf_l1.coef_.ravel(),
}).sort_values("unreg").round(3)


## 20. More practice

### 20.1 Threshold as a cellar rule

`predict` uses 0.5. Sweep `t` in `{0.3, 0.4, 0.5, 0.6, 0.7}` on `clf_best` probabilities. Report F1, false-premium (FP: bad bottled as good) and missed-good (FN).


In [ ]:
proba = clf_best.predict_proba(X_test)[:, 1]
rows = []
for t in (0.30, 0.40, 0.50, 0.60, 0.70):
    pred = (proba >= t).astype(int)
    tp = int(((pred == 1) & (y_test.values == 1)).sum())
    fp = int(((pred == 1) & (y_test.values == 0)).sum())
    fn = int(((pred == 0) & (y_test.values == 1)).sum())
    tn = int(((pred == 0) & (y_test.values == 0)).sum())
    rows.append({"t": t, "F1": f1_score(y_test, pred), "FP_false_premium": fp,
                 "FN_missed_good": fn, "acc": (tp + tn) / len(y_test)})
print(pd.DataFrame(rows).round(3))


### 20.2 Three-test subset vs full card

Refit L2 (`C=0.002`) on only `alcohol`, `volatile acidity`, `total sulfur dioxide`. Compare hold-out F1 to the 11-test model.


In [ ]:
keep = ["alcohol", "volatile acidity", "total sulfur dioxide"]
idx = [list(features.columns).index(c) for c in keep]
sub = LogisticRegression(C=0.002, max_iter=2000)
sub.fit(X_train[:, idx], y_train)
print("3-test F1", f1_score(y_test, sub.predict(X_test[:, idx])))
print("11-test F1", f1_score(y_test, clf_best.predict(X_test)))


### 20.3 Class-weight balanced

The split is nearly even, but try `class_weight='balanced'` at the tuned C. Does recall of class 0 move?


In [ ]:
bal = LogisticRegression(C=gs.best_params_["C"], class_weight="balanced", max_iter=2000)
bal.fit(X_train, y_train)
print(classification_report(y_test, bal.predict(X_test), digits=3))


### 20.4 Elastic-net sketch (optional)

If your sklearn build accepts `penalty='elasticnet'` + `l1_ratio` + `solver='saga'`, try `l1_ratio=0.5` on a small C grid. Otherwise skip and note the deprecation path (`l1_ratio` on the new default estimator).


In [ ]:
try:
    en = LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, C=0.2, max_iter=4000
    )
    en.fit(X_train, y_train)
    print("elastic-net test F1", f1_score(y_test, en.predict(X_test)))
    print(pd.Series(en.coef_.ravel(), index=predictors).round(3))
except Exception as exc:
    print("elastic-net not run:", type(exc).__name__, exc)


## 21. Simulation — edit the knobs

Each replication: optional subsample, optional label flips, optional Gaussian junk columns, then L2 at `C_RIDGE` and L1 at `C_LASSO`. Watch how hold-out F1 and the number of exact-zero L1 weights move.


In [ ]:
# --- knobs ---
C_RIDGE = 0.002
C_LASSO = 0.26
N_SUB = 1279          # ≤ 1599; 1279 matches the lesson train size
FLIP_P = 0.00         # label-noise rate
N_JUNK = 0            # extra N(0,1) columns
N_REPS = 10
SEED = 7
# -------------

rng = np.random.default_rng(SEED)
rows = []
for r in range(N_REPS):
    n = min(N_SUB, len(X))
    idx = rng.choice(len(X), size=n, replace=False)
    Xb = X[idx].copy()
    yb = y.values[idx].copy()
    if FLIP_P > 0:
        flip = rng.random(n) < FLIP_P
        yb[flip] = 1 - yb[flip]
    if N_JUNK > 0:
        Xb = np.hstack([Xb, rng.normal(size=(n, N_JUNK))])
    Xt, Xv, yt, yv = train_test_split(Xb, yb, test_size=0.2, random_state=r)
    ridge = LogisticRegression(C=C_RIDGE, penalty="l2", max_iter=2000)
    ridge.fit(Xt, yt)
    lasso = LogisticRegression(C=C_LASSO, penalty="l1", solver="liblinear", max_iter=2000)
    lasso.fit(Xt, yt)
    rows.append({
        "rep": r,
        "ridge_f1": f1_score(yv, ridge.predict(Xv)),
        "lasso_f1": f1_score(yv, lasso.predict(Xv)),
        "lasso_zeros": int((np.abs(lasso.coef_.ravel()) < 1e-10).sum()),
    })
sim = pd.DataFrame(rows)
print(sim.round(3))
print(sim[["ridge_f1", "lasso_f1", "lasso_zeros"]].agg(["mean", "std"]).round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].boxplot([sim["ridge_f1"], sim["lasso_f1"]], labels=["ridge", "lasso"])
ax[0].set_ylabel("hold-out F1")
ax[0].set_title(f"C_ridge={C_RIDGE}, C_lasso={C_LASSO}, flip={FLIP_P}, junk={N_JUNK}")
ax[1].hist(sim["lasso_zeros"], bins=range(0, 12), color="#1A5276", edgecolor="white")
ax[1].set_xlabel("# exact-zero L1 weights")
ax[1].set_title("sparsity across reps")
plt.tight_layout()
plt.show()


## 22. Audience rewrite

Four cuts of the same result, using the attached audience checklists (data literacy, subject knowledge, time span, expert / technician / executive / nonspecialist).

**Expert (oenologist / statistician).**  
n = 1,599, binary cut at rating 5, prior π̂ = 0.535. Features standardized. Unpenalized MLE and L2 at C = 1 give the same hold-out F1 (0.727) — the L2 ball still contains the MLE. A 5-fold F1 grid on C ∈ [10⁻⁴, 10⁻²] selects C ≈ 0.002 (CV F1 ≈ 0.773); hold-out F1 = 0.735, AUC ≈ 0.81. L1-CV at C ≈ 0.26 zeros **density**, consistent with its linear dependence on alcohol / extract. Coefficients are log-odds per 1 SD. Do not treat the 5-cut as a sensory gold standard.

**Technician (lab / QC).**  
Scale the 11 tests. Fit logistic regression. If you leave C at the software default you have *not* regularized in any useful way. Set C near 0.002 for ridge, or use the L1 path and drop density from the daily card. Alcohol, sulphates, volatile acidity and total SO₂ do the work. F1 on a 320-bottle hold-out is about 0.73 — useful as a screen, not as a release stamp.

**Executive (winery GM / brand).**  
A chemistry-only model flags “good / not-good” at roughly three-in-four F1. Stronger shrinkage (smaller C) beats the default software settings. Density can leave the panel; alcohol and volatile acidity cannot. Use the probability threshold as a cellar rule: lower it to catch more good lots (more false premiums), raise it to protect the reserve label.

**Nonspecialist (curious drinker).**  
We asked 11 lab measurements to guess whether a red wine would have been scored above 5. Alcohol and a clean (not vinegary) profile help. Making the model “shy” about huge coefficients — regularization — stopped it from clinging to a redundant measurement (density) and slightly improved the guess on bottles it had not seen. It is a sorting aid, not a sommelier.


## 23. Takeaways

1. **Scale first.** Regularization penalizes coefficient *size*; unscaled inputs make the penalty meaningless.
2. **Default C = 1 did nothing here.** Train and test F1 matched the unregularized model. The constraint was too loose.
3. **C is inverse strength.** The useful ridge window on this card is about 10⁻³, not 1.
4. **Tune on train CV, confirm on a frozen test fold.** The lesson’s GridSearchCV score is not a test score.
5. **L1 is feature selection.** Density’s coefficient hits zero; alcohol / VA / total SO₂ stay.
6. **F1, not accuracy.** The 53.5% good prior plus a shifted test mix (48%) would flatter a sloppy accuracy headline.
7. **sklearn is moving.** Prefer `penalty=None` or `C=np.inf` today; `penalty='none'` is the lesson spelling.

**Can predict:** a binary good/bad flag from routine red-wine chemistry, with honest F1 around the mid-0.70s.  
**Cannot predict:** a 1–10 sensory score, vintage prestige, price, or white-wine quality from this fit.
